# 19. TP/SL・最大保有時間と失敗したデータ取得
出典: FX (2).ipynb、セルindex [39, 40, 41]。保存出力は results/imported_20260909/ を参照。
研究履歴です。実行順・Notebook内変数・元の価格CSVに依存し、エラーが出たコードも保存しています。
自動判定の文言は元実験の判定であり、監査済みの結論ではありません。全セル一括実行は再現手順ではありません。
[USER_HOME] は匿名化した元のパスです。元Notebook内の案内や依頼文は研究資料として保持しています。


## 元セルindex 39


In [ ]:
# ============================================================
# USD/JPY 15m SIGNAL + 5m EXECUTION
#
# NESTED TP / SL EXIT DIAGNOSTIC
#
# ============================================================
#
# 目的:
#
# 現在のEntry戦略
#
#   RandomForest
#   + Confidence Threshold
#   + Session Selection
#
# を固定したまま、
#
#   TP
#   SL
#
# を追加することで
#
#   Avg Return
#   PF
#   Max DD
#   Cost robustness
#
# が未知年OOSでも改善するか調べる。
#
#
# Signal       : 15m
# Execution    : 5m
# Max Hold     : 30分固定
#
#
# IMPORTANT
# ------------------------------------------------------------
#
# 1. TP/SLによって早くExitしても、
#    今回は追加Entryを許可しない。
#
#    → Entry戦略を変えず、
#      Exitの純粋な効果だけを見る。
#
#
# 2. 同じ5分足の中でTPとSL両方に触れたら、
#    OHLCだけでは順番が分からない。
#
#    → 保守的にSLが先だったものとして扱う。
#
#
# 3. TP/SLはValidationだけで選ぶ。
#
#    Test年では完全固定。
#
# ============================================================


from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score


# ============================================================
# 0. 前コードが実行されているか確認
# ============================================================

REQUIRED_OBJECTS = [

    "data",
    "FEATURES",

    "build_model",
    "predict_frame",

    "choose_threshold",
    "choose_session",

    "select_trades",
    "strategy_stats",

    "BASE_COST",

    "MIN_TRAIN_YEARS",
    "MIN_TRAIN_ROWS",
    "MIN_EVAL_ROWS",

]


missing_objects = [

    name

    for name in REQUIRED_OBJECTS

    if name not in globals()

]


if missing_objects:

    raise RuntimeError(

        "前の PURE SESSION VALUE TEST を先に実行してください。\n"
        f"不足: {missing_objects}"

    )


# ============================================================
# 1. 5分足CSV
# ============================================================

#
# まずこの名前を探す。
#
FIVE_MIN_CSV = (

    Path.cwd()
    / "dukascopy_usdjpy"
    / "usdjpy_5m_2016_2026.csv"

)


# ------------------------------------------------------------
# 見つからない場合、
# 5mらしいCSV候補を表示して停止する。
# ------------------------------------------------------------

if not FIVE_MIN_CSV.exists():

    candidates = [

        p

        for p in Path.cwd().rglob("*.csv")

        if (

            "5m" in p.name.lower()

            and

            (
                "usd" in p.name.lower()
                or
                "jpy" in p.name.lower()
            )

        )

    ]


    print(
        "指定した5分足CSVがありません:"
    )

    print(
        FIVE_MIN_CSV
    )

    print()

    print(
        "見つかった5分足候補:"
    )


    for p in candidates[:20]:

        print(
            " ",
            p
        )


    raise FileNotFoundError(

        "\nFIVE_MIN_CSV を実際の5分足CSVのPathへ変更してください。"

    )


# ============================================================
# 2. CONFIG
# ============================================================

MAX_HOLD_MINUTES = 30


# ------------------------------------------------------------
# ValidationでExitを変更する最低改善幅
#
# 0.001% / trade
#
# 小さすぎる改善ではTP/SLを採用しない。
# ------------------------------------------------------------

MIN_EXIT_IMPROVEMENT = 0.00001


# ------------------------------------------------------------
# 5mデータ最低Coverage
# ------------------------------------------------------------

MIN_EXECUTION_COVERAGE = 0.95


# ------------------------------------------------------------
# Exit Policy
#
# 候補を増やしすぎない。
# ------------------------------------------------------------

EXIT_POLICIES = [

    {
        "name": "BASE_30M",
        "tp": None,
        "sl": None,
    },

    {
        "name": "TP05_SL05",
        "tp": 0.0005,
        "sl": 0.0005,
    },

    {
        "name": "TP08_SL05",
        "tp": 0.0008,
        "sl": 0.0005,
    },

    {
        "name": "TP10_SL05",
        "tp": 0.0010,
        "sl": 0.0005,
    },

    {
        "name": "TP08_SL08",
        "tp": 0.0008,
        "sl": 0.0008,
    },

    {
        "name": "TP10_SL08",
        "tp": 0.0010,
        "sl": 0.0008,
    },

    {
        "name": "TP12_SL08",
        "tp": 0.0012,
        "sl": 0.0008,
    },

    {
        "name": "TP10_SL10",
        "tp": 0.0010,
        "sl": 0.0010,
    },

    {
        "name": "TP15_SL10",
        "tp": 0.0015,
        "sl": 0.0010,
    },

]


COST_LEVELS_EXIT = [

    0.00004,   # 0.004%
    0.00006,   # 0.006%
    0.00008,   # 0.008%
    0.00010,   # 0.010%
    0.00012,   # 0.012%

]


OUTPUT_DIR_EXIT = (

    Path.cwd()
    /
    (
        "nested_exit_5m_"
        +
        datetime.now().strftime(
            "%Y%m%d_%H%M%S"
        )
    )

)


OUTPUT_DIR_EXIT.mkdir(
    exist_ok=False
)


# ============================================================
# 3. 5分足読み込み
# ============================================================

bars5 = pd.read_csv(

    FIVE_MIN_CSV,

    index_col=0,

)


bars5.index = pd.to_datetime(

    bars5.index,

    utc=True,

)


bars5.columns = [

    c.lower()

    for c in bars5.columns

]


required_5m = [

    "open",
    "high",
    "low",
    "close",

]


missing_5m = [

    c

    for c in required_5m

    if c not in bars5.columns

]


if missing_5m:

    raise ValueError(

        f"5分足にOHLC列がありません: {missing_5m}"

    )


bars5 = (

    bars5[
        required_5m
    ]

    .apply(
        pd.to_numeric,
        errors="raise"
    )

    .sort_index()

)


if not bars5.index.is_unique:

    raise ValueError(

        "5分足timestampに重複があります。"

    )


print(
    "===================================="
)

print(
    "5m EXECUTION DATA"
)

print(
    "===================================="
)

print(
    "Rows:",
    len(bars5)
)

print(
    "Period:",
    bars5.index.min(),
    "->",
    bars5.index.max()
)


# ============================================================
# 4. 5分足Path取得
# ============================================================

def get_5m_path(
    entry_time,
):

    #
    # Entryから30分なので、
    #
    # Entry
    # +5
    # +10
    # +15
    # +20
    # +25
    #
    # の6本。
    #

    expected_index = pd.date_range(

        start=
            entry_time,

        periods=
            MAX_HOLD_MINUTES // 5,

        freq=
            "5min",

        tz="UTC",

    )


    path = (

        bars5.reindex(
            expected_index
        )

    )


    if path[
        required_5m
    ].isna().any().any():

        return None


    return path


# ============================================================
# 5. 5mと15mのExit整合性Audit
# ============================================================

def execution_path_audit(
    trades,
):

    checked = 0

    valid = 0

    max_relative_error = 0.0


    for row in trades.itertuples():

        path = get_5m_path(

            row.entry_time

        )


        if path is None:

            continue


        checked += 1


        five_min_timeout_close = (

            path[
                "close"
            ].iloc[-1]

        )


        fifteen_min_exit = (

            row.exit_price

        )


        relative_error = abs(

            five_min_timeout_close

            -

            fifteen_min_exit

        ) / fifteen_min_exit


        max_relative_error = max(

            max_relative_error,

            relative_error,

        )


        #
        # 0.001%以内なら一致扱い
        #
        if relative_error <= 0.00001:

            valid += 1


    coverage = (

        checked
        /
        len(trades)

        if len(trades)

        else np.nan

    )


    consistency = (

        valid
        /
        checked

        if checked

        else np.nan

    )


    return {

        "trades":
            len(trades),

        "paths_found":
            checked,

        "coverage":
            coverage,

        "consistent":
            valid,

        "consistency":
            consistency,

        "max_relative_error":
            max_relative_error,

    }


# ============================================================
# 6. MFE / MAE
# ============================================================

def calculate_mfe_mae(
    direction,
    entry_price,
    path,
):

    highs = (

        path[
            "high"
        ].to_numpy()

    )


    lows = (

        path[
            "low"
        ].to_numpy()

    )


    if direction == "BUY":

        mfe = (

            highs.max()
            /
            entry_price

            -

            1

        )


        mae = (

            lows.min()
            /
            entry_price

            -

            1

        )


    else:

        #
        # SELL
        #
        # Price低下 = 利益
        #

        mfe = (

            1

            -

            lows.min()
            /
            entry_price

        )


        mae = (

            1

            -

            highs.max()
            /
            entry_price

        )


    return (

        mfe,
        mae,

    )


# ============================================================
# 7. 1Tradeを5分足でシミュレーション
# ============================================================

def simulate_trade_exit(
    row,
    policy,
):

    path = get_5m_path(

        row.entry_time

    )


    if path is None:

        return None


    direction = (

        row.direction

    )


    entry = float(

        row.entry_price

    )


    tp = (

        policy[
            "tp"
        ]

    )


    sl = (

        policy[
            "sl"
        ]

    )


    mfe, mae = (

        calculate_mfe_mae(

            direction,
            entry,
            path,

        )

    )


    # --------------------------------------------------------
    # BASELINE = 30分Hold
    # --------------------------------------------------------

    if (

        tp is None

        and

        sl is None

    ):

        exit_price = float(

            path[
                "close"
            ].iloc[-1]

        )


        if direction == "BUY":

            gross_return = (

                exit_price
                /
                entry

                -

                1

            )

        else:

            gross_return = (

                1

                -

                exit_price
                /
                entry

            )


        return {

            "gross_return":
                gross_return,

            "exit_reason":
                "TIMEOUT",

            "exit_time":
                path.index[-1]
                +
                pd.Timedelta(
                    minutes=5
                ),

            "mfe":
                mfe,

            "mae":
                mae,

            "ambiguous":
                False,

        }


    # --------------------------------------------------------
    # TP / SL Price
    # --------------------------------------------------------

    if direction == "BUY":

        tp_price = (

            entry
            *
            (
                1 + tp
            )

        )


        sl_price = (

            entry
            *
            (
                1 - sl
            )

        )


    else:

        tp_price = (

            entry
            *
            (
                1 - tp
            )

        )


        sl_price = (

            entry
            *
            (
                1 + sl
            )

        )


    # --------------------------------------------------------
    # 5分ずつ追う
    # --------------------------------------------------------

    for timestamp, bar in (

        path.iterrows()

    ):


        if direction == "BUY":

            tp_hit = (

                bar[
                    "high"
                ]
                >=
                tp_price

            )


            sl_hit = (

                bar[
                    "low"
                ]
                <=
                sl_price

            )


        else:

            tp_hit = (

                bar[
                    "low"
                ]
                <=
                tp_price

            )


            sl_hit = (

                bar[
                    "high"
                ]
                >=
                sl_price

            )


        # ----------------------------------------------------
        # 同一5分足でTP/SL両方
        #
        # 順番不明なので保守的にSL
        # ----------------------------------------------------

        if tp_hit and sl_hit:

            return {

                "gross_return":
                    -sl,

                "exit_reason":
                    "AMBIG_SL",

                "exit_time":
                    timestamp
                    +
                    pd.Timedelta(
                        minutes=5
                    ),

                "mfe":
                    mfe,

                "mae":
                    mae,

                "ambiguous":
                    True,

            }


        if sl_hit:

            return {

                "gross_return":
                    -sl,

                "exit_reason":
                    "SL",

                "exit_time":
                    timestamp
                    +
                    pd.Timedelta(
                        minutes=5
                    ),

                "mfe":
                    mfe,

                "mae":
                    mae,

                "ambiguous":
                    False,

            }


        if tp_hit:

            return {

                "gross_return":
                    tp,

                "exit_reason":
                    "TP",

                "exit_time":
                    timestamp
                    +
                    pd.Timedelta(
                        minutes=5
                    ),

                "mfe":
                    mfe,

                "mae":
                    mae,

                "ambiguous":
                    False,

            }


    # --------------------------------------------------------
    # 30分経過
    # --------------------------------------------------------

    exit_price = float(

        path[
            "close"
        ].iloc[-1]

    )


    if direction == "BUY":

        gross_return = (

            exit_price
            /
            entry

            -

            1

        )

    else:

        gross_return = (

            1

            -

            exit_price
            /
            entry

        )


    return {

        "gross_return":
            gross_return,

        "exit_reason":
            "TIMEOUT",

        "exit_time":
            path.index[-1]
            +
            pd.Timedelta(
                minutes=5
            ),

        "mfe":
            mfe,

        "mae":
            mae,

        "ambiguous":
            False,

    }


# ============================================================
# 8. 複数TradeへExit Policy適用
# ============================================================

def apply_exit_policy(
    trades,
    policy,
    cost=BASE_COST,
):

    rows = []


    for row in trades.itertuples():

        result = (

            simulate_trade_exit(

                row,
                policy,

            )

        )


        if result is None:

            continue


        rows.append(

            {

                "signal_time":
                    row.Index,

                "entry_time":
                    row.entry_time,

                "direction":
                    row.direction,

                "entry_price":
                    row.entry_price,

                "confidence":
                    row.confidence,

                "test_year":
                    getattr(
                        row,
                        "test_year",
                        np.nan
                    ),

                "threshold":
                    row.threshold,

                "session_policy":
                    row.session_policy,

                "exit_policy":
                    policy[
                        "name"
                    ],

                **result,

            }

        )


    result_frame = (

        pd.DataFrame(
            rows
        )

    )


    if result_frame.empty:

        return result_frame


    result_frame = (

        result_frame

        .set_index(
            "signal_time"
        )

        .sort_index()

    )


    result_frame[
        "net_return"
    ] = (

        result_frame[
            "gross_return"
        ]

        -

        cost

    )


    return result_frame


# ============================================================
# 9. ValidationでExit Policy選択
# ============================================================

def choose_exit_policy(
    validation_trades,
):

    policy_rows = []

    result_by_policy = {}


    for policy in (

        EXIT_POLICIES

    ):

        simulated = (

            apply_exit_policy(

                validation_trades,

                policy,

                cost=
                    BASE_COST,

            )

        )


        if simulated.empty:

            continue


        coverage = (

            len(
                simulated
            )

            /
            len(
                validation_trades
            )

        )


        stats = (

            strategy_stats(

                simulated[
                    "net_return"
                ]

            )

        )


        ambiguous_rate = (

            simulated[
                "ambiguous"
            ].mean()

        )


        policy_rows.append(

            {

                "exit_policy":
                    policy[
                        "name"
                    ],

                "tp":
                    policy[
                        "tp"
                    ],

                "sl":
                    policy[
                        "sl"
                    ],

                "coverage":
                    coverage,

                "ambiguous_rate":
                    ambiguous_rate,

                **stats,

            }

        )


        result_by_policy[
            policy[
                "name"
            ]
        ] = (
            simulated
        )


    table = (

        pd.DataFrame(
            policy_rows
        )

    )


    if table.empty:

        return (

            None,
            table,

        )


    table = (

        table.loc[

            table[
                "coverage"
            ]

            >=

            MIN_EXECUTION_COVERAGE

        ]

        .copy()

    )


    if table.empty:

        return (

            None,
            table,

        )


    # --------------------------------------------------------
    # BASELINE
    # --------------------------------------------------------

    baseline_row = (

        table.loc[

            table[
                "exit_policy"
            ]
            ==
            "BASE_30M"

        ]

    )


    if baseline_row.empty:

        return (

            None,
            table,

        )


    baseline_row = (

        baseline_row.iloc[0]

    )


    baseline_mean = (

        baseline_row[
            "avg_return"
        ]

    )


    baseline_pf = (

        baseline_row[
            "profit_factor"
        ]

    )


    # --------------------------------------------------------
    # best avg return
    # --------------------------------------------------------

    best_row = (

        table.sort_values(

            [
                "avg_return",
                "profit_factor",
            ],

            ascending=[
                False,
                False,
            ],

        )

        .iloc[0]

    )


    # --------------------------------------------------------
    # Conservative Guardrail
    #
    # TP/SL採用条件:
    #
    # 1. Avg ReturnがBaselineより
    #    0.001%以上改善
    #
    # 2. PFもBaseline以上
    #
    # 満たさないならBASE_30Mを維持。
    # --------------------------------------------------------

    if (

        best_row[
            "exit_policy"
        ]
        !=
        "BASE_30M"

        and

        (
            best_row[
                "avg_return"
            ]

            -

            baseline_mean
        )

        >=
        MIN_EXIT_IMPROVEMENT

        and

        best_row[
            "profit_factor"
        ]

        >=

        baseline_pf

    ):

        chosen_name = (

            best_row[
                "exit_policy"
            ]

        )


    else:

        chosen_name = (

            "BASE_30M"

        )


    chosen_policy = next(

        p

        for p in EXIT_POLICIES

        if p[
            "name"
        ]
        ==
        chosen_name

    )


    return (

        chosen_policy,
        table,

    )


# ============================================================
# 10. Nested Walk-Forward
# ============================================================

years = sorted(

    data.index.year.unique()

)


annual_rows = []


fixed_exit_oos_frames = []

selected_exit_oos_frames = []

validation_exit_tables = []

mfe_mae_frames = []


for test_year in years:


    validation_year = (

        test_year
        -
        1

    )


    previous_years = [

        y

        for y in years

        if y
        <
        validation_year

    ]


    if (

        len(
            previous_years
        )
        <
        MIN_TRAIN_YEARS

    ):

        continue


    if validation_year not in years:

        continue


    validation_start = pd.Timestamp(

        year=
            validation_year,

        month=1,

        day=1,

        tz="UTC",

    )


    test_start = pd.Timestamp(

        year=
            test_year,

        month=1,

        day=1,

        tz="UTC",

    )


    test_end = pd.Timestamp(

        year=
            test_year + 1,

        month=1,

        day=1,

        tz="UTC",

    )


    # --------------------------------------------------------
    # Train
    # --------------------------------------------------------

    train = (

        data.loc[

            (
                data.index
                <
                validation_start
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                validation_start
            )

        ]

        .copy()

    )


    # --------------------------------------------------------
    # Validation
    # --------------------------------------------------------

    validation = (

        data.loc[

            (
                data.index
                >=
                validation_start
            )

            &

            (
                data.index
                <
                test_start
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                test_start
            )

        ]

        .copy()

    )


    # --------------------------------------------------------
    # Final Train
    # --------------------------------------------------------

    final_train = (

        data.loc[

            (
                data.index
                <
                test_start
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                test_start
            )

        ]

        .copy()

    )


    # --------------------------------------------------------
    # Test
    # --------------------------------------------------------

    test = (

        data.loc[

            (
                data.index
                >=
                test_start
            )

            &

            (
                data.index
                <
                test_end
            )

            &

            (
                data[
                    "label_end"
                ]
                <=
                test_end
            )

        ]

        .copy()

    )


    if (

        len(train)
        <
        MIN_TRAIN_ROWS

        or

        len(validation)
        <
        MIN_EVAL_ROWS

        or

        len(final_train)
        <
        MIN_TRAIN_ROWS

        or

        len(test)
        <
        MIN_EVAL_ROWS

    ):

        continue


    print()

    print(
        "===================================="
    )

    print(
        f"EXIT TEST YEAR {test_year}"
    )

    print(
        "===================================="
    )


    # ========================================================
    # Model -> Validation
    # ========================================================

    model = (

        build_model()

    )


    model.fit(

        train[
            FEATURES
        ],

        train[
            "target"
        ],

    )


    validation_predictions = (

        predict_frame(

            model,

            validation,

        )

    )


    # ========================================================
    # Threshold
    # ========================================================

    threshold_choice, _ = (

        choose_threshold(

            validation_predictions

        )

    )


    if threshold_choice is None:

        continue


    fixed_threshold = (

        threshold_choice[
            "threshold"
        ]

    )


    # ========================================================
    # Session
    # ========================================================

    session_choice, _ = (

        choose_session(

            validation_predictions,

            fixed_threshold,

        )

    )


    if session_choice is None:

        continue


    selected_session = (

        session_choice[
            "session_policy"
        ]

    )


    # ========================================================
    # Validation Entry Trades
    #
    # Entry scheduleは30分Holdを基準に固定。
    # ========================================================

    validation_trades = (

        select_trades(

            validation_predictions,

            threshold=
                fixed_threshold,

            session_policy=
                selected_session,

            cost=
                BASE_COST,

        )

    )


    # ========================================================
    # 5m Coverage Audit
    # ========================================================

    audit = (

        execution_path_audit(

            validation_trades

        )

    )


    print(
        "Threshold:",
        fixed_threshold
    )

    print(
        "Session:",
        selected_session
    )

    print(
        "5m coverage:",
        round(
            audit[
                "coverage"
            ]
            *
            100,
            2
        ),
        "%"
    )


    if (

        audit[
            "coverage"
        ]

        <
        MIN_EXECUTION_COVERAGE

    ):

        print(
            "5m coverage不足。"
        )

        continue


    # ========================================================
    # Exit policy選択
    # ========================================================

    chosen_exit, exit_table = (

        choose_exit_policy(

            validation_trades

        )

    )


    if chosen_exit is None:

        continue


    exit_table[
        "test_year"
    ] = (
        test_year
    )


    validation_exit_tables.append(

        exit_table

    )


    print(
        "Selected Exit:",
        chosen_exit[
            "name"
        ]
    )


    # ========================================================
    # Final Model
    # ========================================================

    final_model = (

        build_model()

    )


    final_model.fit(

        final_train[
            FEATURES
        ],

        final_train[
            "target"
        ],

    )


    test_predictions = (

        predict_frame(

            final_model,

            test,

        )

    )


    test_auc = (

        roc_auc_score(

            test[
                "target"
            ],

            test_predictions[
                "p_up"
            ],

        )

    )


    # ========================================================
    # Test Entry Trades
    # ========================================================

    test_trades = (

        select_trades(

            test_predictions,

            threshold=
                fixed_threshold,

            session_policy=
                selected_session,

            cost=
                BASE_COST,

        )

    )


    test_trades[
        "test_year"
    ] = (
        test_year
    )


    # ========================================================
    # Fixed 30m
    # ========================================================

    baseline_policy = next(

        p

        for p in EXIT_POLICIES

        if p[
            "name"
        ]
        ==
        "BASE_30M"

    )


    fixed_result = (

        apply_exit_policy(

            test_trades,

            baseline_policy,

            cost=
                BASE_COST,

        )

    )


    # ========================================================
    # Selected Exit
    # ========================================================

    selected_result = (

        apply_exit_policy(

            test_trades,

            chosen_exit,

            cost=
                BASE_COST,

        )

    )


    if (

        fixed_result.empty

        or

        selected_result.empty

    ):

        continue


    fixed_result[
        "test_year"
    ] = (
        test_year
    )


    selected_result[
        "test_year"
    ] = (
        test_year
    )


    fixed_exit_oos_frames.append(

        fixed_result

    )


    selected_exit_oos_frames.append(

        selected_result

    )


    mfe_mae_frames.append(

        selected_result[
            [
                "test_year",
                "direction",
                "mfe",
                "mae",
                "net_return",
                "exit_reason",
            ]
        ].copy()

    )


    fixed_stats = (

        strategy_stats(

            fixed_result[
                "net_return"
            ]

        )

    )


    selected_stats = (

        strategy_stats(

            selected_result[
                "net_return"
            ]

        )

    )


    exit_reason_counts = (

        selected_result[
            "exit_reason"
        ]

        .value_counts()

    )


    ambiguous_rate = (

        selected_result[
            "ambiguous"
        ].mean()

    )


    annual_rows.append(

        {

            "test_year":
                test_year,

            "test_auc":
                test_auc,

            "threshold":
                fixed_threshold,

            "session_policy":
                selected_session,

            "exit_policy":
                chosen_exit[
                    "name"
                ],

            "trades":
                len(
                    selected_result
                ),

            "fixed_avg_return":
                fixed_stats[
                    "avg_return"
                ],

            "fixed_pf":
                fixed_stats[
                    "profit_factor"
                ],

            "fixed_max_dd":
                fixed_stats[
                    "max_dd"
                ],

            "exit_avg_return":
                selected_stats[
                    "avg_return"
                ],

            "exit_pf":
                selected_stats[
                    "profit_factor"
                ],

            "exit_max_dd":
                selected_stats[
                    "max_dd"
                ],

            "tp_count":
                exit_reason_counts.get(
                    "TP",
                    0
                ),

            "sl_count":
                exit_reason_counts.get(
                    "SL",
                    0
                ),

            "ambig_sl_count":
                exit_reason_counts.get(
                    "AMBIG_SL",
                    0
                ),

            "timeout_count":
                exit_reason_counts.get(
                    "TIMEOUT",
                    0
                ),

            "ambiguous_rate":
                ambiguous_rate,

        }

    )


    print(
        "Fixed 30m:",
        "PF",
        round(
            fixed_stats[
                "profit_factor"
            ],
            3
        ),
        "| Avg",
        round(
            fixed_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),
        "%"
    )


    print(
        "Selected:",
        chosen_exit[
            "name"
        ],
        "| PF",
        round(
            selected_stats[
                "profit_factor"
            ],
            3
        ),
        "| Avg",
        round(
            selected_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),
        "%"
    )


# ============================================================
# 11. Annual Results
# ============================================================

annual_exit_results = (

    pd.DataFrame(
        annual_rows
    )

)


print()

print(
    "===================================="
)

print(
    "ANNUAL EXIT RESULTS"
)

print(
    "===================================="
)


annual_show = (

    annual_exit_results.copy()

)


for col in [

    "threshold",

    "fixed_avg_return",
    "exit_avg_return",

    "fixed_max_dd",
    "exit_max_dd",

    "ambiguous_rate",

]:

    if col in annual_show.columns:

        annual_show[
            col
        ] *= (
            100
        )


print(

    annual_show.to_string(
        index=False
    )

)


# ============================================================
# 12. Overall OOS
# ============================================================

all_fixed_exit = (

    pd.concat(
        fixed_exit_oos_frames
    )

    .sort_index()

)


all_selected_exit = (

    pd.concat(
        selected_exit_oos_frames
    )

    .sort_index()

)


fixed_overall = (

    strategy_stats(

        all_fixed_exit[
            "net_return"
        ]

    )

)


exit_overall = (

    strategy_stats(

        all_selected_exit[
            "net_return"
        ]

    )

)


print()

print(
    "===================================="
)

print(
    "OVERALL OOS EXIT COMPARISON"
)

print(
    "===================================="
)


print(
    "FIXED 30M"
)

print(
    fixed_overall
)


print()

print(
    "SELECTED EXIT"
)

print(
    exit_overall
)


# ============================================================
# 13. Exit Policy Selection Frequency
# ============================================================

print()

print(
    "===================================="
)

print(
    "EXIT POLICY SELECTION"
)

print(
    "===================================="
)


print(

    annual_exit_results[
        "exit_policy"
    ]

    .value_counts()

)


# ============================================================
# 14. 年ごとの改善
# ============================================================

annual_exit_results[
    "avg_improved"
] = (

    annual_exit_results[
        "exit_avg_return"
    ]

    >

    annual_exit_results[
        "fixed_avg_return"
    ]

)


annual_exit_results[
    "pf_improved"
] = (

    annual_exit_results[
        "exit_pf"
    ]

    >

    annual_exit_results[
        "fixed_pf"
    ]

)


annual_exit_results[
    "dd_improved"
] = (

    annual_exit_results[
        "exit_max_dd"
    ]

    >

    annual_exit_results[
        "fixed_max_dd"
    ]

)


print()

print(
    "===================================="
)

print(
    "EXIT VALUE"
)

print(
    "===================================="
)


print(
    "Avg improved:",
    annual_exit_results[
        "avg_improved"
    ].sum(),
    "/",
    len(
        annual_exit_results
    )
)


print(
    "PF improved:",
    annual_exit_results[
        "pf_improved"
    ].sum(),
    "/",
    len(
        annual_exit_results
    )
)


print(
    "DD improved:",
    annual_exit_results[
        "dd_improved"
    ].sum(),
    "/",
    len(
        annual_exit_results
    )
)


# ============================================================
# 15. MFE / MAE
# ============================================================

mfe_mae = (

    pd.concat(
        mfe_mae_frames
    )

    .sort_index()

)


def percentile_summary(
    series
):

    return pd.Series(

        {

            "count":
                series.count(),

            "p10":
                series.quantile(
                    0.10
                ),

            "p25":
                series.quantile(
                    0.25
                ),

            "median":
                series.quantile(
                    0.50
                ),

            "p75":
                series.quantile(
                    0.75
                ),

            "p90":
                series.quantile(
                    0.90
                ),

        }

    )


mfe_summary = (

    percentile_summary(

        mfe_mae[
            "mfe"
        ]

    )

)


mae_summary = (

    percentile_summary(

        mfe_mae[
            "mae"
        ]

    )

)


print()

print(
    "===================================="
)

print(
    "MFE"
)

print(
    "===================================="
)

print(

    mfe_summary
    *
    pd.Series(
        {
            "count": 1,
            "p10": 100,
            "p25": 100,
            "median": 100,
            "p75": 100,
            "p90": 100,
        }
    )

)


print()

print(
    "===================================="
)

print(
    "MAE"
)

print(
    "===================================="
)

print(

    mae_summary
    *
    pd.Series(
        {
            "count": 1,
            "p10": 100,
            "p25": 100,
            "median": 100,
            "p75": 100,
            "p90": 100,
        }
    )

)


# ============================================================
# 16. Exit reason
# ============================================================

print()

print(
    "===================================="
)

print(
    "EXIT REASONS"
)

print(
    "===================================="
)


print(

    all_selected_exit[
        "exit_reason"
    ]

    .value_counts()

)


print()

print(
    "Ambiguous rate:"
)

print(

    all_selected_exit[
        "ambiguous"
    ].mean()
    *
    100,

    "%"

)


# ============================================================
# 17. Cost Stress
# ============================================================

cost_rows = []


for strategy_name, frame in [

    (
        "FIXED_30M",
        all_fixed_exit,
    ),

    (
        "SELECTED_EXIT",
        all_selected_exit,
    ),

]:


    for cost in (

        COST_LEVELS_EXIT

    ):


        returns = (

            frame[
                "gross_return"
            ]

            -

            cost

        )


        stats = (

            strategy_stats(
                returns
            )

        )


        cost_rows.append(

            {

                "strategy":
                    strategy_name,

                "cost_pct":
                    cost
                    *
                    100,

                **stats,

            }

        )


exit_cost_results = (

    pd.DataFrame(
        cost_rows
    )

)


cost_show = (

    exit_cost_results.copy()

)


for col in [

    "win_rate",
    "avg_return",
    "median_return",
    "max_dd",
    "growth",

]:

    if col in cost_show.columns:

        cost_show[
            col
        ] *= (
            100
        )


print()

print(
    "===================================="
)

print(
    "EXIT COST STRESS"
)

print(
    "===================================="
)


print(

    cost_show.to_string(
        index=False
    )

)


# ============================================================
# 18. Automatic Decision
# ============================================================

print()

print(
    "===================================="
)

print(
    "AUTOMATIC DECISION"
)

print(
    "===================================="
)


n_years = (

    len(
        annual_exit_results
    )

)


avg_better_years = (

    annual_exit_results[
        "avg_improved"
    ].sum()

)


pf_better_years = (

    annual_exit_results[
        "pf_improved"
    ].sum()

)


print(
    "Fixed PF:",
    fixed_overall[
        "profit_factor"
    ]
)


print(
    "Exit PF:",
    exit_overall[
        "profit_factor"
    ]
)


print(
    "Fixed Avg:",
    fixed_overall[
        "avg_return"
    ]
    *
    100,
    "%"
)


print(
    "Exit Avg:",
    exit_overall[
        "avg_return"
    ]
    *
    100,
    "%"
)


print(
    "PF better years:",
    pf_better_years,
    "/",
    n_years
)


print(
    "Avg better years:",
    avg_better_years,
    "/",
    n_years
)


print()


if (

    exit_overall[
        "profit_factor"
    ]

    >

    fixed_overall[
        "profit_factor"
    ]

    and

    exit_overall[
        "avg_return"
    ]

    >

    fixed_overall[
        "avg_return"
    ]

    and

    pf_better_years
    >=
    4

    and

    avg_better_years
    >=
    4

):

    print(
        "判定: TP/SL HAS CLEAR OOS VALUE"
    )

    print(
        "TP/SLを正式候補として次段階へ進めます。"
    )

    print(
        "次はMax HoldやATRベースExitを検証する価値があります。"
    )


else:

    print(
        "判定: TP/SL DOES NOT YET ADD CLEAR OOS VALUE"
    )

    print(
        "現在の30分固定Exitを維持する方が合理的です。"
    )

    print(
        "MFE/MAEを使って次のExit候補を再設計します。"
    )


# ============================================================
# 19. Graph - Annual PF
# ============================================================

plt.figure(
    figsize=(
        9,
        5
    )
)


plt.plot(

    annual_exit_results[
        "test_year"
    ],

    annual_exit_results[
        "fixed_pf"
    ],

    marker="o",

    label=
        "Fixed 30m",

)


plt.plot(

    annual_exit_results[
        "test_year"
    ],

    annual_exit_results[
        "exit_pf"
    ],

    marker="o",

    label=
        "TP/SL",

)


plt.axhline(
    1,
    linewidth=1
)


plt.xlabel(
    "Test Year"
)

plt.ylabel(
    "Profit Factor"
)

plt.title(
    "Nested OOS Exit Comparison"
)

plt.legend()

plt.tight_layout()

plt.show()


# ============================================================
# 20. Save
# ============================================================

annual_exit_results.to_csv(

    OUTPUT_DIR_EXIT
    /
    "annual_exit_results.csv",

    index=False,

)


all_fixed_exit.to_csv(

    OUTPUT_DIR_EXIT
    /
    "fixed_30m_oos.csv",

)


all_selected_exit.to_csv(

    OUTPUT_DIR_EXIT
    /
    "selected_exit_oos.csv",

)


mfe_mae.to_csv(

    OUTPUT_DIR_EXIT
    /
    "mfe_mae.csv",

)


exit_cost_results.to_csv(

    OUTPUT_DIR_EXIT
    /
    "exit_cost_stress.csv",

    index=False,

)


if validation_exit_tables:

    pd.concat(

        validation_exit_tables,

        ignore_index=True,

    ).to_csv(

        OUTPUT_DIR_EXIT
        /
        "validation_exit_search.csv",

        index=False,

    )


print()

print(
    "===================================="
)

print(
    "FINISHED"
)

print(
    "===================================="
)


print(

    OUTPUT_DIR_EXIT.resolve()

)


print()

print(
    "結果で見せてほしい場所:"
)

print(
    "1. ANNUAL EXIT RESULTS"
)

print(
    "2. OVERALL OOS EXIT COMPARISON"
)

print(
    "3. EXIT POLICY SELECTION"
)

print(
    "4. EXIT VALUE"
)

print(
    "5. MFE / MAE"
)

print(
    "6. EXIT REASONS"
)

print(
    "7. EXIT COST STRESS"
)

print(
    "8. AUTOMATIC DECISION"
)


## 元セルindex 40


In [ ]:
# ============================================================
# USD/JPY EXIT TEST v2
#
# Robust TP / SL Test
#
# 15m Signal
# 5m Execution if available
# 15m conservative fallback if 5m history is insufficient
#
# ============================================================


from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score


# ============================================================
# 0. 前コード確認
# ============================================================

REQUIRED_OBJECTS = [
    "data",
    "df",
    "FEATURES",
    "build_model",
    "predict_frame",
    "choose_threshold",
    "choose_session",
    "select_trades",
    "strategy_stats",
    "BASE_COST",
    "MIN_TRAIN_YEARS",
    "MIN_TRAIN_ROWS",
    "MIN_EVAL_ROWS",
]

missing = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]

if missing:
    print("前の PURE SESSION VALUE TEST の変数が不足しています。")
    print(missing)
    print()
    print("前のコードを先に実行してください。")
    raise RuntimeError("Previous session-test cell is required.")


# ============================================================
# 1. CONFIG
# ============================================================

MAX_HOLD_MINUTES = 30

MIN_EXIT_IMPROVEMENT = 0.00001

BASE_EXECUTION_COST = BASE_COST


EXIT_POLICIES = [
    {
        "name": "BASE_30M",
        "tp": None,
        "sl": None,
    },
    {
        "name": "TP05_SL05",
        "tp": 0.0005,
        "sl": 0.0005,
    },
    {
        "name": "TP08_SL05",
        "tp": 0.0008,
        "sl": 0.0005,
    },
    {
        "name": "TP10_SL05",
        "tp": 0.0010,
        "sl": 0.0005,
    },
    {
        "name": "TP08_SL08",
        "tp": 0.0008,
        "sl": 0.0008,
    },
    {
        "name": "TP10_SL08",
        "tp": 0.0010,
        "sl": 0.0008,
    },
    {
        "name": "TP12_SL08",
        "tp": 0.0012,
        "sl": 0.0008,
    },
    {
        "name": "TP10_SL10",
        "tp": 0.0010,
        "sl": 0.0010,
    },
    {
        "name": "TP15_SL10",
        "tp": 0.0015,
        "sl": 0.0010,
    },
]


COST_LEVELS_EXIT = [
    0.00004,
    0.00006,
    0.00008,
    0.00010,
    0.00012,
]


OUTPUT_DIR_EXIT = (
    Path.cwd()
    /
    (
        "nested_exit_v2_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
    )
)

OUTPUT_DIR_EXIT.mkdir(
    exist_ok=False
)


# ============================================================
# 2. 5分足CSVを自動検索
# ============================================================

print()
print("=" * 60)
print("5分足データを自動検索します")
print("=" * 60)


SEARCH_ROOTS = [
    Path.cwd(),
    Path.home() / "dukascopy_usdjpy",
    Path.home() / "fx_experiment_runs",
    Path.home() / "Documents" / "Codex",
]


candidate_paths = []


for root in SEARCH_ROOTS:

    if not root.exists():
        continue

    try:

        for p in root.rglob("*.csv"):

            name = p.name.lower()

            if (
                "5m" in name
                and
                (
                    "usdjpy" in name
                    or
                    "usd_jpy" in name
                )
            ):

                candidate_paths.append(
                    p
                )

    except Exception as e:

        print(
            "検索をスキップ:",
            root,
            e
        )


candidate_paths = list(
    dict.fromkeys(
        candidate_paths
    )
)


print(
    "候補数:",
    len(candidate_paths)
)


for p in candidate_paths[:30]:

    print(
        " ",
        p
    )


# ============================================================
# 3. CSVを安全に読む
# ============================================================

def safe_read_ohlc_csv(path):

    try:

        temp = pd.read_csv(
            path,
            index_col=0
        )

        temp.index = pd.to_datetime(
            temp.index,
            utc=True,
            errors="coerce"
        )

        temp = temp.loc[
            temp.index.notna()
        ].copy()

        temp.columns = [
            str(c).lower()
            for c in temp.columns
        ]


        required = [
            "open",
            "high",
            "low",
            "close",
        ]


        if not all(
            c in temp.columns
            for c in required
        ):

            return None


        temp = (
            temp[required]
            .apply(
                pd.to_numeric,
                errors="coerce"
            )
            .dropna()
            .sort_index()
        )


        if len(temp) == 0:
            return None


        return temp


    except Exception as e:

        print(
            "読込失敗:",
            path
        )

        print(
            "理由:",
            e
        )

        return None


# ============================================================
# 4. 見つかった5分足を全部確認
# ============================================================

loaded_5m = []


for p in candidate_paths:

    temp = safe_read_ohlc_csv(
        p
    )


    if temp is None:
        continue


    loaded_5m.append(
        (
            p,
            temp
        )
    )


    print()
    print(
        "OK:",
        p
    )

    print(
        "Rows:",
        len(temp)
    )

    print(
        "Period:",
        temp.index.min(),
        "->",
        temp.index.max()
    )


# ============================================================
# 5. 5分足を結合
# ============================================================

if loaded_5m:

    frames = [
        frame
        for _, frame
        in loaded_5m
    ]


    bars5 = (
        pd.concat(
            frames
        )
        .sort_index()
    )


    # 同じtimestampが複数ファイルにある場合
    # 最後のものを残す
    bars5 = (
        bars5.loc[
            ~bars5.index.duplicated(
                keep="last"
            )
        ]
        .sort_index()
    )


    print()
    print("=" * 60)
    print("5分足を結合しました")
    print("=" * 60)

    print(
        "Rows:",
        len(bars5)
    )

    print(
        "Period:",
        bars5.index.min(),
        "->",
        bars5.index.max()
    )


else:

    bars5 = None

    print()
    print(
        "利用可能な5分足CSVが見つかりませんでした。"
    )


# ============================================================
# 6. Execution Mode判定
#
# 完全な10年分5mがなくても、
# 個別Tradeごとに5mがあれば5mを使う。
#
# なければ15mへFallback。
# ============================================================

if bars5 is not None:

    EXECUTION_MODE = "HYBRID_5M_15M"

else:

    EXECUTION_MODE = "15M_FALLBACK"


print()
print(
    "Execution mode:",
    EXECUTION_MODE
)


# ============================================================
# 7. 15分足Execution Path
# ============================================================

def get_15m_path(entry_time):

    expected = pd.date_range(
        start=entry_time,
        periods=2,
        freq="15min",
        tz="UTC",
    )


    path = df.reindex(
        expected
    )


    required = [
        "open",
        "high",
        "low",
        "close",
    ]


    if path[
        required
    ].isna().any().any():

        return None


    return path[
        required
    ].copy()


# ============================================================
# 8. 5分足Execution Path
# ============================================================

def get_5m_path(entry_time):

    if bars5 is None:
        return None


    expected = pd.date_range(
        start=entry_time,
        periods=6,
        freq="5min",
        tz="UTC",
    )


    path = bars5.reindex(
        expected
    )


    required = [
        "open",
        "high",
        "low",
        "close",
    ]


    if path[
        required
    ].isna().any().any():

        return None


    return path[
        required
    ].copy()


# ============================================================
# 9. TradeごとにExecution pathを決める
# ============================================================

def get_execution_path(entry_time):

    # 最優先 = 5分足
    path5 = get_5m_path(
        entry_time
    )


    if path5 is not None:

        return (
            path5,
            5,
            "5M"
        )


    # なければ15分足
    path15 = get_15m_path(
        entry_time
    )


    if path15 is not None:

        return (
            path15,
            15,
            "15M"
        )


    return (
        None,
        None,
        None
    )


# ============================================================
# 10. MFE / MAE
# ============================================================

def calculate_mfe_mae(
    direction,
    entry_price,
    path,
):

    highs = (
        path["high"]
        .to_numpy()
    )

    lows = (
        path["low"]
        .to_numpy()
    )


    if direction == "BUY":

        mfe = (
            highs.max()
            /
            entry_price
            -
            1
        )

        mae = (
            lows.min()
            /
            entry_price
            -
            1
        )


    else:

        mfe = (
            1
            -
            lows.min()
            /
            entry_price
        )

        mae = (
            1
            -
            highs.max()
            /
            entry_price
        )


    return (
        mfe,
        mae
    )


# ============================================================
# 11. Trade simulator
# ============================================================

def simulate_trade_exit(
    row,
    policy,
):

    path, bar_minutes, execution_source = (
        get_execution_path(
            row.entry_time
        )
    )


    if path is None:

        return None


    direction = (
        row.direction
    )

    entry = float(
        row.entry_price
    )


    tp = policy["tp"]
    sl = policy["sl"]


    mfe, mae = (
        calculate_mfe_mae(
            direction,
            entry,
            path,
        )
    )


    # ========================================================
    # BASELINE 30分
    # ========================================================

    if (
        tp is None
        and
        sl is None
    ):

        exit_price = float(
            path["close"].iloc[-1]
        )


        if direction == "BUY":

            gross_return = (
                exit_price
                /
                entry
                -
                1
            )

        else:

            gross_return = (
                1
                -
                exit_price
                /
                entry
            )


        return {
            "gross_return":
                gross_return,

            "exit_reason":
                "TIMEOUT",

            "exit_time":
                path.index[-1]
                +
                pd.Timedelta(
                    minutes=bar_minutes
                ),

            "mfe":
                mfe,

            "mae":
                mae,

            "ambiguous":
                False,

            "execution_source":
                execution_source,
        }


    # ========================================================
    # TP / SL price
    # ========================================================

    if direction == "BUY":

        tp_price = (
            entry
            *
            (
                1 + tp
            )
        )

        sl_price = (
            entry
            *
            (
                1 - sl
            )
        )


    else:

        tp_price = (
            entry
            *
            (
                1 - tp
            )
        )

        sl_price = (
            entry
            *
            (
                1 + sl
            )
        )


    # ========================================================
    # BarごとにTP / SL判定
    # ========================================================

    for timestamp, bar in path.iterrows():


        if direction == "BUY":

            tp_hit = (
                bar["high"]
                >=
                tp_price
            )

            sl_hit = (
                bar["low"]
                <=
                sl_price
            )


        else:

            tp_hit = (
                bar["low"]
                <=
                tp_price
            )

            sl_hit = (
                bar["high"]
                >=
                sl_price
            )


        # ----------------------------------------------------
        # 同じbar内でTP・SL両方
        #
        # 順序不明なのでSL先。
        # かなり保守的。
        # ----------------------------------------------------

        if tp_hit and sl_hit:

            return {
                "gross_return":
                    -sl,

                "exit_reason":
                    "AMBIG_SL",

                "exit_time":
                    timestamp
                    +
                    pd.Timedelta(
                        minutes=bar_minutes
                    ),

                "mfe":
                    mfe,

                "mae":
                    mae,

                "ambiguous":
                    True,

                "execution_source":
                    execution_source,
            }


        if sl_hit:

            return {
                "gross_return":
                    -sl,

                "exit_reason":
                    "SL",

                "exit_time":
                    timestamp
                    +
                    pd.Timedelta(
                        minutes=bar_minutes
                    ),

                "mfe":
                    mfe,

                "mae":
                    mae,

                "ambiguous":
                    False,

                "execution_source":
                    execution_source,
            }


        if tp_hit:

            return {
                "gross_return":
                    tp,

                "exit_reason":
                    "TP",

                "exit_time":
                    timestamp
                    +
                    pd.Timedelta(
                        minutes=bar_minutes
                    ),

                "mfe":
                    mfe,

                "mae":
                    mae,

                "ambiguous":
                    False,

                "execution_source":
                    execution_source,
            }


    # ========================================================
    # TP/SLなし → 30分終了
    # ========================================================

    exit_price = float(
        path["close"].iloc[-1]
    )


    if direction == "BUY":

        gross_return = (
            exit_price
            /
            entry
            -
            1
        )

    else:

        gross_return = (
            1
            -
            exit_price
            /
            entry
        )


    return {
        "gross_return":
            gross_return,

        "exit_reason":
            "TIMEOUT",

        "exit_time":
            path.index[-1]
            +
            pd.Timedelta(
                minutes=bar_minutes
            ),

        "mfe":
            mfe,

        "mae":
            mae,

        "ambiguous":
            False,

        "execution_source":
            execution_source,
    }


# ============================================================
# 12. Exit Policy適用
# ============================================================

def apply_exit_policy(
    trades,
    policy,
    cost=BASE_EXECUTION_COST,
):

    rows = []


    for row in trades.itertuples():

        result = (
            simulate_trade_exit(
                row,
                policy,
            )
        )


        if result is None:
            continue


        rows.append(
            {
                "signal_time":
                    row.Index,

                "entry_time":
                    row.entry_time,

                "direction":
                    row.direction,

                "entry_price":
                    row.entry_price,

                "confidence":
                    row.confidence,

                "test_year":
                    getattr(
                        row,
                        "test_year",
                        np.nan
                    ),

                "threshold":
                    row.threshold,

                "session_policy":
                    row.session_policy,

                "exit_policy":
                    policy["name"],

                **result,
            }
        )


    result_frame = pd.DataFrame(
        rows
    )


    if result_frame.empty:
        return result_frame


    result_frame = (
        result_frame
        .set_index(
            "signal_time"
        )
        .sort_index()
    )


    result_frame[
        "net_return"
    ] = (
        result_frame[
            "gross_return"
        ]
        -
        cost
    )


    return result_frame


# ============================================================
# 13. Exit Policy Validation
# ============================================================

def choose_exit_policy(
    validation_trades,
):

    rows = []


    for policy in EXIT_POLICIES:

        simulated = (
            apply_exit_policy(
                validation_trades,
                policy,
                cost=BASE_EXECUTION_COST,
            )
        )


        if simulated.empty:
            continue


        coverage = (
            len(simulated)
            /
            len(validation_trades)
        )


        stats = (
            strategy_stats(
                simulated[
                    "net_return"
                ]
            )
        )


        rows.append(
            {
                "exit_policy":
                    policy["name"],

                "tp":
                    policy["tp"],

                "sl":
                    policy["sl"],

                "coverage":
                    coverage,

                "ambiguous_rate":
                    simulated[
                        "ambiguous"
                    ].mean(),

                **stats,
            }
        )


    table = pd.DataFrame(
        rows
    )


    if table.empty:

        return (
            EXIT_POLICIES[0],
            table
        )


    baseline_rows = table.loc[
        table["exit_policy"]
        ==
        "BASE_30M"
    ]


    if baseline_rows.empty:

        best_name = (
            table
            .sort_values(
                "avg_return",
                ascending=False
            )
            .iloc[0]["exit_policy"]
        )


        chosen = next(
            p
            for p in EXIT_POLICIES
            if p["name"] == best_name
        )


        return (
            chosen,
            table
        )


    baseline = (
        baseline_rows.iloc[0]
    )


    best = (
        table.sort_values(
            [
                "avg_return",
                "profit_factor",
            ],
            ascending=[
                False,
                False,
            ]
        )
        .iloc[0]
    )


    # ========================================================
    # TP/SL採用には明確なValidation改善を要求
    # ========================================================

    if (
        best["exit_policy"]
        !=
        "BASE_30M"

        and

        (
            best["avg_return"]
            -
            baseline["avg_return"]
        )
        >=
        MIN_EXIT_IMPROVEMENT

        and

        best["profit_factor"]
        >=
        baseline["profit_factor"]
    ):

        chosen_name = (
            best["exit_policy"]
        )

    else:

        chosen_name = (
            "BASE_30M"
        )


    chosen = next(
        p
        for p in EXIT_POLICIES
        if p["name"] == chosen_name
    )


    return (
        chosen,
        table
    )


# ============================================================
# 14. Nested Walk-Forward
# ============================================================

years = sorted(
    data.index.year.unique()
)


annual_rows = []

fixed_frames = []
selected_frames = []

validation_exit_tables = []


for test_year in years:


    validation_year = (
        test_year - 1
    )


    previous_years = [
        y
        for y in years
        if y < validation_year
    ]


    if (
        len(previous_years)
        <
        MIN_TRAIN_YEARS
    ):
        continue


    if validation_year not in years:
        continue


    validation_start = pd.Timestamp(
        year=validation_year,
        month=1,
        day=1,
        tz="UTC"
    )


    test_start = pd.Timestamp(
        year=test_year,
        month=1,
        day=1,
        tz="UTC"
    )


    test_end = pd.Timestamp(
        year=test_year + 1,
        month=1,
        day=1,
        tz="UTC"
    )


    # ========================================================
    # Train
    # ========================================================

    train = data.loc[
        (
            data.index
            <
            validation_start
        )
        &
        (
            data["label_end"]
            <=
            validation_start
        )
    ].copy()


    # ========================================================
    # Validation
    # ========================================================

    validation = data.loc[
        (
            data.index
            >=
            validation_start
        )
        &
        (
            data.index
            <
            test_start
        )
        &
        (
            data["label_end"]
            <=
            test_start
        )
    ].copy()


    # ========================================================
    # Final Train
    # ========================================================

    final_train = data.loc[
        (
            data.index
            <
            test_start
        )
        &
        (
            data["label_end"]
            <=
            test_start
        )
    ].copy()


    # ========================================================
    # Test
    # ========================================================

    test = data.loc[
        (
            data.index
            >=
            test_start
        )
        &
        (
            data.index
            <
            test_end
        )
        &
        (
            data["label_end"]
            <=
            test_end
        )
    ].copy()


    if (
        len(train) < MIN_TRAIN_ROWS
        or
        len(validation) < MIN_EVAL_ROWS
        or
        len(final_train) < MIN_TRAIN_ROWS
        or
        len(test) < MIN_EVAL_ROWS
    ):

        continue


    print()
    print("=" * 60)
    print(
        "TEST YEAR",
        test_year
    )
    print("=" * 60)


    # ========================================================
    # Train -> Validation
    # ========================================================

    model = build_model()


    model.fit(
        train[FEATURES],
        train["target"],
    )


    validation_predictions = (
        predict_frame(
            model,
            validation,
        )
    )


    # ========================================================
    # Threshold
    # ========================================================

    threshold_choice, _ = (
        choose_threshold(
            validation_predictions
        )
    )


    if threshold_choice is None:
        continue


    threshold = (
        threshold_choice[
            "threshold"
        ]
    )


    # ========================================================
    # Session
    # ========================================================

    session_choice, _ = (
        choose_session(
            validation_predictions,
            threshold,
        )
    )


    if session_choice is None:
        continue


    session = (
        session_choice[
            "session_policy"
        ]
    )


    validation_trades = (
        select_trades(
            validation_predictions,
            threshold=threshold,
            session_policy=session,
            cost=BASE_COST,
        )
    )


    # ========================================================
    # ExitをValidationで決定
    # ========================================================

    chosen_exit, exit_table = (
        choose_exit_policy(
            validation_trades
        )
    )


    exit_table[
        "test_year"
    ] = test_year


    validation_exit_tables.append(
        exit_table
    )


    print(
        "Threshold:",
        threshold
    )

    print(
        "Session:",
        session
    )

    print(
        "Exit:",
        chosen_exit["name"]
    )


    # ========================================================
    # Test Model
    # ========================================================

    final_model = build_model()


    final_model.fit(
        final_train[FEATURES],
        final_train["target"],
    )


    test_predictions = (
        predict_frame(
            final_model,
            test,
        )
    )


    test_auc = roc_auc_score(
        test["target"],
        test_predictions["p_up"],
    )


    test_trades = (
        select_trades(
            test_predictions,
            threshold=threshold,
            session_policy=session,
            cost=BASE_COST,
        )
    )


    test_trades[
        "test_year"
    ] = test_year


    # ========================================================
    # Fixed 30m
    # ========================================================

    base_policy = next(
        p
        for p in EXIT_POLICIES
        if p["name"] == "BASE_30M"
    )


    fixed = (
        apply_exit_policy(
            test_trades,
            base_policy,
        )
    )


    selected = (
        apply_exit_policy(
            test_trades,
            chosen_exit,
        )
    )


    if (
        fixed.empty
        or
        selected.empty
    ):
        continue


    fixed[
        "test_year"
    ] = test_year


    selected[
        "test_year"
    ] = test_year


    fixed_frames.append(
        fixed
    )

    selected_frames.append(
        selected
    )


    fixed_stats = strategy_stats(
        fixed["net_return"]
    )


    selected_stats = strategy_stats(
        selected["net_return"]
    )


    source_counts = (
        selected[
            "execution_source"
        ]
        .value_counts()
    )


    annual_rows.append(
        {
            "test_year":
                test_year,

            "test_auc":
                test_auc,

            "threshold":
                threshold,

            "session_policy":
                session,

            "exit_policy":
                chosen_exit["name"],

            "trades":
                len(selected),

            "five_min_trades":
                source_counts.get(
                    "5M",
                    0
                ),

            "fifteen_min_trades":
                source_counts.get(
                    "15M",
                    0
                ),

            "fixed_avg":
                fixed_stats[
                    "avg_return"
                ],

            "fixed_pf":
                fixed_stats[
                    "profit_factor"
                ],

            "fixed_dd":
                fixed_stats[
                    "max_dd"
                ],

            "exit_avg":
                selected_stats[
                    "avg_return"
                ],

            "exit_pf":
                selected_stats[
                    "profit_factor"
                ],

            "exit_dd":
                selected_stats[
                    "max_dd"
                ],

            "ambiguous_rate":
                selected[
                    "ambiguous"
                ].mean(),
        }
    )


    print(
        "Fixed PF:",
        round(
            fixed_stats[
                "profit_factor"
            ],
            3
        )
    )


    print(
        "Exit PF:",
        round(
            selected_stats[
                "profit_factor"
            ],
            3
        )
    )


    print(
        "Fixed Avg:",
        round(
            fixed_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),
        "%"
    )


    print(
        "Exit Avg:",
        round(
            selected_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),
        "%"
    )


    print(
        "5m trades:",
        source_counts.get(
            "5M",
            0
        )
    )


    print(
        "15m fallback:",
        source_counts.get(
            "15M",
            0
        )
    )


# ============================================================
# 15. 結果
# ============================================================

annual_exit = pd.DataFrame(
    annual_rows
)


print()
print("=" * 60)
print("ANNUAL EXIT RESULTS")
print("=" * 60)

print(
    annual_exit.to_string(
        index=False
    )
)


# ============================================================
# 16. Overall
# ============================================================

all_fixed = (
    pd.concat(
        fixed_frames
    )
    .sort_index()
)


all_selected = (
    pd.concat(
        selected_frames
    )
    .sort_index()
)


fixed_overall = strategy_stats(
    all_fixed[
        "net_return"
    ]
)


selected_overall = strategy_stats(
    all_selected[
        "net_return"
    ]
)


print()
print("=" * 60)
print("OVERALL OOS EXIT")
print("=" * 60)


print(
    "FIXED 30M"
)

print(
    fixed_overall
)


print()

print(
    "SELECTED EXIT"
)

print(
    selected_overall
)


# ============================================================
# 17. 5m利用率
# ============================================================

print()
print("=" * 60)
print("EXECUTION SOURCE")
print("=" * 60)


print(
    all_selected[
        "execution_source"
    ]
    .value_counts()
)


five_min_ratio = (
    (
        all_selected[
            "execution_source"
        ]
        ==
        "5M"
    )
    .mean()
)


print()

print(
    "5m使用率:",
    five_min_ratio
    *
    100,
    "%"
)


# ============================================================
# 18. Exit reasons
# ============================================================

print()
print("=" * 60)
print("EXIT REASONS")
print("=" * 60)


print(
    all_selected[
        "exit_reason"
    ]
    .value_counts()
)


print()

print(
    "Ambiguous rate:",
    all_selected[
        "ambiguous"
    ]
    .mean()
    *
    100,
    "%"
)


# ============================================================
# 19. MFE / MAE
# ============================================================

def show_percentiles(
    series,
    name,
):

    print()
    print(name)

    for q in [
        0.10,
        0.25,
        0.50,
        0.75,
        0.90,
    ]:

        print(
            f"{int(q*100)}%:",
            series.quantile(q)
            *
            100,
            "%"
        )


show_percentiles(
    all_selected[
        "mfe"
    ],
    "MFE"
)


show_percentiles(
    all_selected[
        "mae"
    ],
    "MAE"
)


# ============================================================
# 20. Cost Stress
# ============================================================

cost_rows = []


for strategy_name, frame in [
    (
        "FIXED_30M",
        all_fixed
    ),
    (
        "SELECTED_EXIT",
        all_selected
    ),
]:

    for cost in COST_LEVELS_EXIT:

        returns = (
            frame[
                "gross_return"
            ]
            -
            cost
        )


        stats = strategy_stats(
            returns
        )


        cost_rows.append(
            {
                "strategy":
                    strategy_name,

                "cost_pct":
                    cost
                    *
                    100,

                **stats
            }
        )


cost_results = pd.DataFrame(
    cost_rows
)


print()
print("=" * 60)
print("EXIT COST STRESS")
print("=" * 60)


print(
    cost_results.to_string(
        index=False
    )
)


# ============================================================
# 21. OOS改善判定
# ============================================================

annual_exit[
    "pf_improved"
] = (
    annual_exit[
        "exit_pf"
    ]
    >
    annual_exit[
        "fixed_pf"
    ]
)


annual_exit[
    "avg_improved"
] = (
    annual_exit[
        "exit_avg"
    ]
    >
    annual_exit[
        "fixed_avg"
    ]
)


annual_exit[
    "dd_improved"
] = (
    annual_exit[
        "exit_dd"
    ]
    >
    annual_exit[
        "fixed_dd"
    ]
)


print()
print("=" * 60)
print("EXIT VALUE")
print("=" * 60)


print(
    "PF improved:",
    annual_exit[
        "pf_improved"
    ].sum(),
    "/",
    len(
        annual_exit
    )
)


print(
    "Avg improved:",
    annual_exit[
        "avg_improved"
    ].sum(),
    "/",
    len(
        annual_exit
    )
)


print(
    "DD improved:",
    annual_exit[
        "dd_improved"
    ].sum(),
    "/",
    len(
        annual_exit
    )
)


# ============================================================
# 22. Automatic Decision
# ============================================================

print()
print("=" * 60)
print("AUTOMATIC DECISION")
print("=" * 60)


print(
    "Fixed PF:",
    fixed_overall[
        "profit_factor"
    ]
)


print(
    "Exit PF:",
    selected_overall[
        "profit_factor"
    ]
)


print(
    "Fixed Avg:",
    fixed_overall[
        "avg_return"
    ]
    *
    100,
    "%"
)


print(
    "Exit Avg:",
    selected_overall[
        "avg_return"
    ]
    *
    100,
    "%"
)


pf_years = (
    annual_exit[
        "pf_improved"
    ].sum()
)


avg_years = (
    annual_exit[
        "avg_improved"
    ].sum()
)


if (
    selected_overall[
        "profit_factor"
    ]
    >
    fixed_overall[
        "profit_factor"
    ]

    and

    selected_overall[
        "avg_return"
    ]
    >
    fixed_overall[
        "avg_return"
    ]

    and

    pf_years >= 4

    and

    avg_years >= 4
):

    print()
    print(
        "判定: TP/SL HAS CLEAR OOS VALUE"
    )

    print(
        "TP/SLは次段階へ進める価値があります。"
    )

else:

    print()
    print(
        "判定: TP/SL DOES NOT YET ADD CLEAR OOS VALUE"
    )

    print(
        "30分固定Exitを維持し、MFE/MAEからExit候補を再設計します。"
    )


# ============================================================
# 23. Save
# ============================================================

annual_exit.to_csv(
    OUTPUT_DIR_EXIT
    /
    "annual_exit_results.csv",

    index=False
)


all_fixed.to_csv(
    OUTPUT_DIR_EXIT
    /
    "fixed_exit_oos.csv"
)


all_selected.to_csv(
    OUTPUT_DIR_EXIT
    /
    "selected_exit_oos.csv"
)


cost_results.to_csv(
    OUTPUT_DIR_EXIT
    /
    "exit_cost_stress.csv",

    index=False
)


if validation_exit_tables:

    pd.concat(
        validation_exit_tables,
        ignore_index=True
    ).to_csv(
        OUTPUT_DIR_EXIT
        /
        "validation_exit_search.csv",

        index=False
    )


print()
print("=" * 60)
print("FINISHED")
print("=" * 60)

print(
    OUTPUT_DIR_EXIT.resolve()
)


print()
print(
    "今回見てほしいもの:"
)

print(
    "1. ANNUAL EXIT RESULTS"
)

print(
    "2. OVERALL OOS EXIT"
)

print(
    "3. EXECUTION SOURCE"
)

print(
    "4. EXIT REASONS"
)

print(
    "5. MFE / MAE"
)

print(
    "6. EXIT COST STRESS"
)

print(
    "7. EXIT VALUE"
)

print(
    "8. AUTOMATIC DECISION"
)


## 元セルindex 41


In [ ]:
# ============================================================
# USD/JPY
# NESTED MAX HOLD DIAGNOSTIC
#
# 目的:
#
# 現在完成しているEntry側
#
# RandomForest
# + Confidence Threshold
# + Session Selection
#
# を変更せず、
#
# 「何分保有するのが良いのか？」
#
# を長期15分足データで検証する。
#
#
# 正式なNested選択:
#   15分
#   30分
#
# 診断のみ:
#   45分
#   60分
#
#
# 重要:
# 45/60分は既存Entry間隔だとポジション重複が起きうるため、
# 今回は正式な戦略選択には使わない。
#
# ============================================================


from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import roc_auc_score


# ============================================================
# 0. 前コード確認
# ============================================================

REQUIRED_OBJECTS = [
    "data",
    "FEATURES",
    "build_model",
    "predict_frame",
    "choose_threshold",
    "choose_session",
    "select_trades",
    "strategy_stats",
    "BASE_COST",
    "MIN_TRAIN_YEARS",
    "MIN_TRAIN_ROWS",
    "MIN_EVAL_ROWS",
]


missing = [
    name
    for name in REQUIRED_OBJECTS
    if name not in globals()
]


if missing:

    print("前のPURE SESSION VALUE TESTの変数が不足しています。")
    print("不足:", missing)

    raise RuntimeError(
        "前のSession検証コードを先に実行してください。"
    )


# ============================================================
# 1. CONFIG
# ============================================================

# ------------------------------------------------------------
# 正式にValidationで選択する候補
#
# 15分なら早めに決済
# 30分は現在のBaseline
# ------------------------------------------------------------

SELECTION_HOLDS = [
    15,
    30,
]


# ------------------------------------------------------------
# OOS診断だけする候補
#
# 45/60分が明確に強いなら、
# 次の実験でEntry scheduleも含めて正式検証する。
# ------------------------------------------------------------

DIAGNOSTIC_HOLDS = [
    15,
    30,
    45,
    60,
]


# ------------------------------------------------------------
# 15分を採用するための最低改善幅
#
# 0.001% / trade
# ------------------------------------------------------------

MIN_HOLD_IMPROVEMENT = 0.00001


# ------------------------------------------------------------
# ExitデータCoverage
# ------------------------------------------------------------

MIN_COVERAGE = 0.98


# ------------------------------------------------------------
# Baseline再現の許容誤差
#
# 0.005%
# ------------------------------------------------------------

AUDIT_TOLERANCE = 0.00005


COST_LEVELS = [
    0.00004,   # 0.004%
    0.00006,
    0.00008,
    0.00010,
    0.00012,
]


OUTPUT_DIR = (
    Path.cwd()
    /
    (
        "max_hold_nested_"
        + datetime.now().strftime("%Y%m%d_%H%M%S")
    )
)


OUTPUT_DIR.mkdir(
    exist_ok=False
)


# ============================================================
# 2. 15分足OHLCデータを自動検出
#
# Notebook内に存在するDataFrameから
#
# open
# high
# low
# close
#
# を持ち、
# 15分刻みに近い最長DataFrameを探す。
# ============================================================

def normalize_ohlc_candidate(obj):

    if not isinstance(obj, pd.DataFrame):
        return None


    temp = obj.copy()


    temp.columns = [
        str(c).lower()
        for c in temp.columns
    ]


    required = [
        "open",
        "high",
        "low",
        "close",
    ]


    if not all(
        c in temp.columns
        for c in required
    ):

        return None


    try:

        temp.index = pd.to_datetime(
            temp.index,
            utc=True,
            errors="coerce",
        )

    except Exception:

        return None


    temp = temp.loc[
        temp.index.notna()
    ].copy()


    if len(temp) < 1000:

        return None


    temp = (
        temp[required]
        .apply(
            pd.to_numeric,
            errors="coerce"
        )
        .dropna()
        .sort_index()
    )


    temp = temp.loc[
        ~temp.index.duplicated(
            keep="last"
        )
    ]


    if len(temp) < 1000:

        return None


    diffs = (
        temp.index
        .to_series()
        .diff()
        .dropna()
        .dt.total_seconds()
        /
        60
    )


    if len(diffs) == 0:

        return None


    fifteen_ratio = (
        (
            (diffs >= 14)
            &
            (diffs <= 16)
        )
        .mean()
    )


    return {
        "frame": temp,
        "rows": len(temp),
        "fifteen_ratio": fifteen_ratio,
    }


ohlc_candidates = []


for name, obj in list(globals().items()):

    result = normalize_ohlc_candidate(
        obj
    )


    if result is None:
        continue


    ohlc_candidates.append(
        {
            "name": name,
            **result,
        }
    )


if not ohlc_candidates:

    raise RuntimeError(
        "Notebook内に15分足OHLC DataFrameが見つかりません。"
    )


# 15分刻み率が高いものを優先し、
# 次に行数が多いものを選ぶ

ohlc_candidates = sorted(
    ohlc_candidates,
    key=lambda x: (
        x["fifteen_ratio"],
        x["rows"],
    ),
    reverse=True,
)


chosen_ohlc = ohlc_candidates[0]


price15 = chosen_ohlc[
    "frame"
].copy()


print()
print("=" * 60)
print("15M PRICE DATA")
print("=" * 60)

print(
    "使用DataFrame:",
    chosen_ohlc["name"]
)

print(
    "Rows:",
    len(price15)
)

print(
    "15分刻み率:",
    chosen_ohlc["fifteen_ratio"] * 100,
    "%"
)

print(
    "Period:",
    price15.index.min(),
    "->",
    price15.index.max()
)


# ============================================================
# 3. 指定時間後の価格を取得
# ============================================================

def get_hold_path(
    entry_time,
    hold_minutes,
):

    entry_time = pd.Timestamp(
        entry_time
    )


    if entry_time.tzinfo is None:

        entry_time = entry_time.tz_localize(
            "UTC"
        )

    else:

        entry_time = entry_time.tz_convert(
            "UTC"
        )


    if hold_minutes % 15 != 0:

        raise ValueError(
            "hold_minutesは15分単位にしてください。"
        )


    bars = (
        hold_minutes
        //
        15
    )


    expected = pd.date_range(
        start=entry_time,
        periods=bars,
        freq="15min",
        tz="UTC",
    )


    path = price15.reindex(
        expected
    )


    if path[
        [
            "open",
            "high",
            "low",
            "close",
        ]
    ].isna().any().any():

        return None


    return path


# ============================================================
# 4. 1TradeのHold Return
# ============================================================

def simulate_hold_trade(
    row,
    hold_minutes,
):

    path = get_hold_path(
        row.entry_time,
        hold_minutes,
    )


    if path is None:

        return None


    entry_price = float(
        row.entry_price
    )


    exit_price = float(
        path["close"].iloc[-1]
    )


    direction = row.direction


    if direction == "BUY":

        gross_return = (
            exit_price
            /
            entry_price
            -
            1
        )


    elif direction == "SELL":

        gross_return = (
            1
            -
            exit_price
            /
            entry_price
        )


    else:

        return None


    return {
        "gross_return":
            gross_return,

        "exit_price":
            exit_price,

        "exit_time":
            path.index[-1]
            +
            pd.Timedelta(
                minutes=15
            ),
    }


# ============================================================
# 5. Trade集合へHold適用
# ============================================================

def apply_hold_policy(
    trades,
    hold_minutes,
    cost=BASE_COST,
):

    rows = []


    for row in trades.itertuples():

        result = simulate_hold_trade(
            row,
            hold_minutes,
        )


        if result is None:
            continue


        rows.append(
            {
                "signal_time":
                    row.Index,

                "entry_time":
                    row.entry_time,

                "entry_price":
                    row.entry_price,

                "direction":
                    row.direction,

                "confidence":
                    row.confidence,

                "threshold":
                    getattr(
                        row,
                        "threshold",
                        np.nan,
                    ),

                "session_policy":
                    getattr(
                        row,
                        "session_policy",
                        "UNKNOWN",
                    ),

                "test_year":
                    getattr(
                        row,
                        "test_year",
                        np.nan,
                    ),

                "hold_minutes":
                    hold_minutes,

                **result,
            }
        )


    frame = pd.DataFrame(
        rows
    )


    if frame.empty:

        return frame


    frame = (
        frame
        .set_index(
            "signal_time"
        )
        .sort_index()
    )


    frame[
        "net_return"
    ] = (
        frame[
            "gross_return"
        ]
        -
        cost
    )


    return frame


# ============================================================
# 6. 30分Baseline再現Audit
#
# 以前select_trades()が計算した30分Returnと、
# 今回15分足から再構築した30分Returnが一致するか。
# ============================================================

def audit_30m_reconstruction(
    trades,
):

    reconstructed = (
        apply_hold_policy(
            trades,
            30,
            cost=BASE_COST,
        )
    )


    if reconstructed.empty:

        return {
            "trades": len(trades),
            "matched": 0,
            "coverage": 0.0,
            "median_abs_diff": np.nan,
            "max_abs_diff": np.nan,
            "within_tolerance": np.nan,
        }


    if "gross_return" in trades.columns:

        original = (
            trades[
                "gross_return"
            ]
        )


    elif "net_return" in trades.columns:

        original = (
            trades[
                "net_return"
            ]
            +
            BASE_COST
        )


    else:

        return {
            "trades": len(trades),
            "matched": len(reconstructed),
            "coverage": (
                len(reconstructed)
                /
                len(trades)
            ),
            "median_abs_diff": np.nan,
            "max_abs_diff": np.nan,
            "within_tolerance": np.nan,
        }


    comparison = pd.DataFrame(
        {
            "original":
                original,

            "reconstructed":
                reconstructed[
                    "gross_return"
                ],
        }
    ).dropna()


    if comparison.empty:

        return {
            "trades": len(trades),
            "matched": 0,
            "coverage": 0.0,
            "median_abs_diff": np.nan,
            "max_abs_diff": np.nan,
            "within_tolerance": np.nan,
        }


    comparison[
        "abs_diff"
    ] = abs(
        comparison["original"]
        -
        comparison["reconstructed"]
    )


    return {
        "trades":
            len(trades),

        "matched":
            len(comparison),

        "coverage":
            len(comparison)
            /
            len(trades),

        "median_abs_diff":
            comparison[
                "abs_diff"
            ].median(),

        "max_abs_diff":
            comparison[
                "abs_diff"
            ].max(),

        "within_tolerance":
            (
                comparison[
                    "abs_diff"
                ]
                <=
                AUDIT_TOLERANCE
            )
            .mean(),
    }


# ============================================================
# 7. ValidationでHold選択
# ============================================================

def choose_hold(
    validation_trades,
):

    rows = []


    for hold in SELECTION_HOLDS:

        frame = apply_hold_policy(
            validation_trades,
            hold,
            cost=BASE_COST,
        )


        if frame.empty:
            continue


        coverage = (
            len(frame)
            /
            len(validation_trades)
        )


        if coverage < MIN_COVERAGE:
            continue


        stats = strategy_stats(
            frame["net_return"]
        )


        rows.append(
            {
                "hold_minutes":
                    hold,

                "coverage":
                    coverage,

                **stats,
            }
        )


    table = pd.DataFrame(
        rows
    )


    if table.empty:

        return (
            30,
            table,
        )


    baseline_rows = table.loc[
        table["hold_minutes"]
        ==
        30
    ]


    if baseline_rows.empty:

        return (
            30,
            table,
        )


    baseline = baseline_rows.iloc[0]


    best = (
        table
        .sort_values(
            [
                "avg_return",
                "profit_factor",
            ],
            ascending=[
                False,
                False,
            ],
        )
        .iloc[0]
    )


    # --------------------------------------------------------
    # 15分を採用するには
    #
    # ・平均Returnが30分より明確に上
    # ・PFも30分以上
    #
    # を要求する。
    # --------------------------------------------------------

    if (
        best["hold_minutes"] != 30

        and

        (
            best["avg_return"]
            -
            baseline["avg_return"]
        )
        >=
        MIN_HOLD_IMPROVEMENT

        and

        best["profit_factor"]
        >=
        baseline["profit_factor"]
    ):

        chosen_hold = int(
            best["hold_minutes"]
        )


    else:

        chosen_hold = 30


    return (
        chosen_hold,
        table,
    )


# ============================================================
# 8. Nested Walk-Forward
# ============================================================

years = sorted(
    data.index.year.unique()
)


annual_rows = []

audit_rows = []

validation_tables = []

baseline_frames = []

selected_frames = []

oos_entry_frames = []

diagnostic_rows = []


for test_year in years:


    validation_year = (
        test_year - 1
    )


    previous_years = [
        y
        for y in years
        if y < validation_year
    ]


    if (
        len(previous_years)
        <
        MIN_TRAIN_YEARS
    ):

        continue


    if validation_year not in years:

        continue


    validation_start = pd.Timestamp(
        year=validation_year,
        month=1,
        day=1,
        tz="UTC",
    )


    test_start = pd.Timestamp(
        year=test_year,
        month=1,
        day=1,
        tz="UTC",
    )


    test_end = pd.Timestamp(
        year=test_year + 1,
        month=1,
        day=1,
        tz="UTC",
    )


    # ========================================================
    # Train
    # ========================================================

    train = data.loc[
        (
            data.index
            <
            validation_start
        )
        &
        (
            data["label_end"]
            <=
            validation_start
        )
    ].copy()


    # ========================================================
    # Validation
    # ========================================================

    validation = data.loc[
        (
            data.index
            >=
            validation_start
        )
        &
        (
            data.index
            <
            test_start
        )
        &
        (
            data["label_end"]
            <=
            test_start
        )
    ].copy()


    # ========================================================
    # Final Train
    # ========================================================

    final_train = data.loc[
        (
            data.index
            <
            test_start
        )
        &
        (
            data["label_end"]
            <=
            test_start
        )
    ].copy()


    # ========================================================
    # Test
    # ========================================================

    test = data.loc[
        (
            data.index
            >=
            test_start
        )
        &
        (
            data.index
            <
            test_end
        )
        &
        (
            data["label_end"]
            <=
            test_end
        )
    ].copy()


    if (
        len(train) < MIN_TRAIN_ROWS
        or
        len(validation) < MIN_EVAL_ROWS
        or
        len(final_train) < MIN_TRAIN_ROWS
        or
        len(test) < MIN_EVAL_ROWS
    ):

        continue


    print()
    print("=" * 60)
    print(
        "TEST YEAR",
        test_year
    )
    print("=" * 60)


    # ========================================================
    # Model -> Validation
    # ========================================================

    model = build_model()


    model.fit(
        train[FEATURES],
        train["target"],
    )


    validation_predictions = predict_frame(
        model,
        validation,
    )


    # ========================================================
    # Threshold
    # ========================================================

    threshold_choice, _ = choose_threshold(
        validation_predictions
    )


    if threshold_choice is None:
        continue


    threshold = threshold_choice[
        "threshold"
    ]


    # ========================================================
    # Session
    # ========================================================

    session_choice, _ = choose_session(
        validation_predictions,
        threshold,
    )


    if session_choice is None:
        continue


    session = session_choice[
        "session_policy"
    ]


    # ========================================================
    # Validation Trades
    # ========================================================

    validation_trades = select_trades(
        validation_predictions,
        threshold=threshold,
        session_policy=session,
        cost=BASE_COST,
    )


    if len(validation_trades) == 0:
        continue


    # ========================================================
    # 30m Audit
    # ========================================================

    audit = audit_30m_reconstruction(
        validation_trades
    )


    audit_rows.append(
        {
            "test_year":
                test_year,

            **audit,
        }
    )


    print(
        "30m reconstruction match:",
        (
            audit["within_tolerance"]
            * 100
            if pd.notna(
                audit["within_tolerance"]
            )
            else np.nan
        ),
        "%"
    )


    # ========================================================
    # Hold選択
    # ========================================================

    selected_hold, hold_table = choose_hold(
        validation_trades
    )


    if not hold_table.empty:

        hold_table[
            "test_year"
        ] = test_year


        validation_tables.append(
            hold_table
        )


    print(
        "Threshold:",
        threshold
    )

    print(
        "Session:",
        session
    )

    print(
        "Selected Hold:",
        selected_hold,
        "minutes"
    )


    # ========================================================
    # Final Model -> Test
    # ========================================================

    final_model = build_model()


    final_model.fit(
        final_train[FEATURES],
        final_train["target"],
    )


    test_predictions = predict_frame(
        final_model,
        test,
    )


    test_auc = roc_auc_score(
        test["target"],
        test_predictions["p_up"],
    )


    test_trades = select_trades(
        test_predictions,
        threshold=threshold,
        session_policy=session,
        cost=BASE_COST,
    )


    if len(test_trades) == 0:
        continue


    test_trades = test_trades.copy()


    test_trades[
        "test_year"
    ] = test_year


    oos_entry_frames.append(
        test_trades
    )


    # ========================================================
    # Baseline 30m
    # ========================================================

    baseline = apply_hold_policy(
        test_trades,
        30,
        cost=BASE_COST,
    )


    # ========================================================
    # Selected Hold
    # ========================================================

    selected = apply_hold_policy(
        test_trades,
        selected_hold,
        cost=BASE_COST,
    )


    if (
        baseline.empty
        or
        selected.empty
    ):

        continue


    baseline[
        "test_year"
    ] = test_year


    selected[
        "test_year"
    ] = test_year


    baseline_frames.append(
        baseline
    )


    selected_frames.append(
        selected
    )


    baseline_stats = strategy_stats(
        baseline["net_return"]
    )


    selected_stats = strategy_stats(
        selected["net_return"]
    )


    # ========================================================
    # 15/30/45/60 OOS診断
    # ========================================================

    for diagnostic_hold in DIAGNOSTIC_HOLDS:

        diagnostic = apply_hold_policy(
            test_trades,
            diagnostic_hold,
            cost=BASE_COST,
        )


        if diagnostic.empty:
            continue


        diag_stats = strategy_stats(
            diagnostic[
                "net_return"
            ]
        )


        diagnostic_rows.append(
            {
                "test_year":
                    test_year,

                "hold_minutes":
                    diagnostic_hold,

                "trades":
                    len(diagnostic),

                "coverage":
                    len(diagnostic)
                    /
                    len(test_trades),

                **diag_stats,
            }
        )


    annual_rows.append(
        {
            "test_year":
                test_year,

            "test_auc":
                test_auc,

            "threshold":
                threshold,

            "session_policy":
                session,

            "selected_hold":
                selected_hold,

            "trades":
                len(selected),

            "baseline_avg":
                baseline_stats[
                    "avg_return"
                ],

            "baseline_pf":
                baseline_stats[
                    "profit_factor"
                ],

            "baseline_dd":
                baseline_stats[
                    "max_dd"
                ],

            "selected_avg":
                selected_stats[
                    "avg_return"
                ],

            "selected_pf":
                selected_stats[
                    "profit_factor"
                ],

            "selected_dd":
                selected_stats[
                    "max_dd"
                ],
        }
    )


    print(
        "30m PF:",
        round(
            baseline_stats[
                "profit_factor"
            ],
            3
        ),
        "| Avg:",
        round(
            baseline_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),
        "%"
    )


    print(
        str(selected_hold) + "m PF:",
        round(
            selected_stats[
                "profit_factor"
            ],
            3
        ),
        "| Avg:",
        round(
            selected_stats[
                "avg_return"
            ]
            *
            100,
            5
        ),
        "%"
    )


# ============================================================
# 9. DataFrames
# ============================================================

annual_results = pd.DataFrame(
    annual_rows
)


audit_results = pd.DataFrame(
    audit_rows
)


diagnostic_results = pd.DataFrame(
    diagnostic_rows
)


# ============================================================
# 10. 30分再現Audit
# ============================================================

print()
print("=" * 60)
print("30M RECONSTRUCTION AUDIT")
print("=" * 60)


if not audit_results.empty:

    audit_show = audit_results.copy()


    for col in [
        "coverage",
        "within_tolerance",
        "median_abs_diff",
        "max_abs_diff",
    ]:

        if col in audit_show.columns:

            audit_show[col] *= 100


    print(
        audit_show.to_string(
            index=False
        )
    )


    mean_match = audit_results[
        "within_tolerance"
    ].mean()


    print()
    print(
        "平均一致率:",
        mean_match * 100,
        "%"
    )


    if (
        pd.notna(mean_match)
        and
        mean_match >= 0.95
    ):

        print(
            "判定: 15分足からの30分Exit再構築は概ね正常です。"
        )

    else:

        print(
            "注意: 30分Baselineとのズレがあります。"
        )

        print(
            "この場合はExit最適化より先にtimestamp/価格定義を再確認します。"
        )


# ============================================================
# 11. Annual Nested結果
# ============================================================

print()
print("=" * 60)
print("ANNUAL MAX HOLD RESULTS")
print("=" * 60)


annual_show = annual_results.copy()


for col in [
    "threshold",
    "baseline_avg",
    "selected_avg",
    "baseline_dd",
    "selected_dd",
]:

    if col in annual_show.columns:
        annual_show[col] *= 100


print(
    annual_show.to_string(
        index=False
    )
)


# ============================================================
# 12. Overall Nested OOS
# ============================================================

all_baseline = (
    pd.concat(
        baseline_frames
    )
    .sort_index()
)


all_selected = (
    pd.concat(
        selected_frames
    )
    .sort_index()
)


baseline_overall = strategy_stats(
    all_baseline[
        "net_return"
    ]
)


selected_overall = strategy_stats(
    all_selected[
        "net_return"
    ]
)


print()
print("=" * 60)
print("OVERALL NESTED OOS")
print("=" * 60)


print("BASELINE 30M")
print(
    baseline_overall
)


print()
print("SELECTED HOLD")
print(
    selected_overall
)


# ============================================================
# 13. Hold選択頻度
# ============================================================

print()
print("=" * 60)
print("HOLD SELECTION FREQUENCY")
print("=" * 60)


print(
    annual_results[
        "selected_hold"
    ]
    .value_counts()
    .sort_index()
)


# ============================================================
# 14. 年ごとの改善
# ============================================================

annual_results[
    "pf_improved"
] = (
    annual_results[
        "selected_pf"
    ]
    >
    annual_results[
        "baseline_pf"
    ]
)


annual_results[
    "avg_improved"
] = (
    annual_results[
        "selected_avg"
    ]
    >
    annual_results[
        "baseline_avg"
    ]
)


annual_results[
    "dd_improved"
] = (
    annual_results[
        "selected_dd"
    ]
    >
    annual_results[
        "baseline_dd"
    ]
)


print()
print("=" * 60)
print("MAX HOLD VALUE")
print("=" * 60)


print(
    "PF improved:",
    annual_results[
        "pf_improved"
    ].sum(),
    "/",
    len(annual_results)
)


print(
    "Avg improved:",
    annual_results[
        "avg_improved"
    ].sum(),
    "/",
    len(annual_results)
)


print(
    "DD improved:",
    annual_results[
        "dd_improved"
    ].sum(),
    "/",
    len(annual_results)
)


# ============================================================
# 15. OOS Hold Curve
#
# これはTest結果を後からまとめた診断。
#
# 45/60分は正式選択ではない。
# ============================================================

print()
print("=" * 60)
print("OOS HOLD CURVE - DIAGNOSTIC ONLY")
print("=" * 60)


hold_curve_rows = []


for hold in DIAGNOSTIC_HOLDS:

    subset = diagnostic_results.loc[
        diagnostic_results[
            "hold_minutes"
        ]
        ==
        hold
    ]


    if subset.empty:
        continue


    # 各年のReturnを単純平均するのではなく、
    # 実際の全OOS tradeを再評価する

    all_entries = pd.concat(
        oos_entry_frames
    )


    hold_frame = apply_hold_policy(
        all_entries,
        hold,
        cost=BASE_COST,
    )


    if hold_frame.empty:
        continue


    stats = strategy_stats(
        hold_frame[
            "net_return"
        ]
    )


    hold_curve_rows.append(
        {
            "hold_minutes":
                hold,

            "trades":
                len(hold_frame),

            **stats,
        }
    )


hold_curve = pd.DataFrame(
    hold_curve_rows
)


print(
    hold_curve.to_string(
        index=False
    )
)


print()
print(
    "注意:"
)

print(
    "45/60分はEntryが重複する可能性があるため、"
)

print(
    "この表で強くてもまだ正式採用はしません。"
)


# ============================================================
# 16. 年別Hold診断
# ============================================================

print()
print("=" * 60)
print("YEAR x HOLD")
print("=" * 60)


if not diagnostic_results.empty:

    diag_show = diagnostic_results.copy()


    diag_show[
        "avg_return"
    ] *= 100


    diag_show[
        "max_dd"
    ] *= 100


    print(
        diag_show[
            [
                "test_year",
                "hold_minutes",
                "trades",
                "avg_return",
                "profit_factor",
                "max_dd",
            ]
        ].to_string(
            index=False
        )
    )


# ============================================================
# 17. Cost Stress
# ============================================================

cost_rows = []


for strategy_name, frame in [
    (
        "BASELINE_30M",
        all_baseline,
    ),
    (
        "SELECTED_HOLD",
        all_selected,
    ),
]:

    for cost in COST_LEVELS:

        returns = (
            frame[
                "gross_return"
            ]
            -
            cost
        )


        stats = strategy_stats(
            returns
        )


        cost_rows.append(
            {
                "strategy":
                    strategy_name,

                "cost_pct":
                    cost * 100,

                **stats,
            }
        )


cost_results = pd.DataFrame(
    cost_rows
)


print()
print("=" * 60)
print("MAX HOLD COST STRESS")
print("=" * 60)


print(
    cost_results.to_string(
        index=False
    )
)


# ============================================================
# 18. Automatic Decision
# ============================================================

print()
print("=" * 60)
print("AUTOMATIC DECISION")
print("=" * 60)


n_years = len(
    annual_results
)


pf_better = annual_results[
    "pf_improved"
].sum()


avg_better = annual_results[
    "avg_improved"
].sum()


print(
    "30m PF:",
    baseline_overall[
        "profit_factor"
    ]
)


print(
    "Selected PF:",
    selected_overall[
        "profit_factor"
    ]
)


print(
    "30m Avg Return:",
    baseline_overall[
        "avg_return"
    ]
    * 100,
    "%"
)


print(
    "Selected Avg Return:",
    selected_overall[
        "avg_return"
    ]
    * 100,
    "%"
)


print(
    "PF better years:",
    pf_better,
    "/",
    n_years
)


print(
    "Avg better years:",
    avg_better,
    "/",
    n_years
)


print()


if (
    selected_overall[
        "profit_factor"
    ]
    >
    baseline_overall[
        "profit_factor"
    ]

    and

    selected_overall[
        "avg_return"
    ]
    >
    baseline_overall[
        "avg_return"
    ]

    and

    pf_better >= 4

    and

    avg_better >= 4
):

    print(
        "判定: SHORTER MAX HOLD HAS CLEAR OOS VALUE"
    )

    print(
        "15分Exitを正式候補として残します。"
    )


else:

    print(
        "判定: 30M BASELINE REMAINS STRONG"
    )

    print(
        "現時点では30分固定Exitを維持する方が合理的です。"
    )


# ============================================================
# 19. 45/60分診断
# ============================================================

if not hold_curve.empty:

    baseline_curve = hold_curve.loc[
        hold_curve[
            "hold_minutes"
        ]
        ==
        30
    ]


    longer = hold_curve.loc[
        hold_curve[
            "hold_minutes"
        ]
        >
        30
    ]


    if (
        not baseline_curve.empty
        and
        not longer.empty
    ):

        baseline_pf = baseline_curve.iloc[0][
            "profit_factor"
        ]


        baseline_avg = baseline_curve.iloc[0][
            "avg_return"
        ]


        longer_best = (
            longer
            .sort_values(
                [
                    "avg_return",
                    "profit_factor",
                ],
                ascending=[
                    False,
                    False,
                ]
            )
            .iloc[0]
        )


        print()
        print("=" * 60)
        print("LONG HOLD DIAGNOSTIC")
        print("=" * 60)


        print(
            "Best long hold:",
            int(
                longer_best[
                    "hold_minutes"
                ]
            ),
            "minutes"
        )


        print(
            "PF:",
            longer_best[
                "profit_factor"
            ]
        )


        print(
            "Avg Return:",
            longer_best[
                "avg_return"
            ]
            *
            100,
            "%"
        )


        if (
            longer_best[
                "profit_factor"
            ]
            >
            baseline_pf

            and

            longer_best[
                "avg_return"
            ]
            >
            baseline_avg
        ):

            print()
            print(
                "45/60分側に改善シグナルがあります。"
            )

            print(
                "次はポジション重複を正しく処理したLong Hold検証を行う価値があります。"
            )


        else:

            print()
            print(
                "45/60分に明確な改善はありません。"
            )

            print(
                "Exit時間探索はここで終了してよい可能性が高いです。"
            )


# ============================================================
# 20. Graph
# ============================================================

if not hold_curve.empty:

    plt.figure(
        figsize=(
            8,
            5
        )
    )


    plt.plot(
        hold_curve[
            "hold_minutes"
        ],
        hold_curve[
            "profit_factor"
        ],
        marker="o",
    )


    plt.axhline(
        1,
        linewidth=1,
    )


    plt.xlabel(
        "Hold Minutes"
    )


    plt.ylabel(
        "Profit Factor"
    )


    plt.title(
        "OOS Hold-Time Diagnostic"
    )


    plt.tight_layout()

    plt.show()


# ============================================================
# 21. Save
# ============================================================

annual_results.to_csv(
    OUTPUT_DIR
    /
    "annual_max_hold.csv",

    index=False,
)


audit_results.to_csv(
    OUTPUT_DIR
    /
    "30m_reconstruction_audit.csv",

    index=False,
)


diagnostic_results.to_csv(
    OUTPUT_DIR
    /
    "year_hold_diagnostic.csv",

    index=False,
)


hold_curve.to_csv(
    OUTPUT_DIR
    /
    "overall_hold_curve.csv",

    index=False,
)


cost_results.to_csv(
    OUTPUT_DIR
    /
    "max_hold_cost_stress.csv",

    index=False,
)


all_baseline.to_csv(
    OUTPUT_DIR
    /
    "baseline_30m_oos.csv",
)


all_selected.to_csv(
    OUTPUT_DIR
    /
    "selected_hold_oos.csv",
)


if validation_tables:

    pd.concat(
        validation_tables,
        ignore_index=True,
    ).to_csv(
        OUTPUT_DIR
        /
        "validation_hold_search.csv",

        index=False,
    )


print()
print("=" * 60)
print("FINISHED")
print("=" * 60)

print(
    OUTPUT_DIR.resolve()
)


print()
print("結果で見たい場所:")

print(
    "1. 30M RECONSTRUCTION AUDIT"
)

print(
    "2. ANNUAL MAX HOLD RESULTS"
)

print(
    "3. OVERALL NESTED OOS"
)

print(
    "4. HOLD SELECTION FREQUENCY"
)

print(
    "5. MAX HOLD VALUE"
)

print(
    "6. OOS HOLD CURVE"
)

print(
    "7. YEAR x HOLD"
)

print(
    "8. MAX HOLD COST STRESS"
)

print(
    "9. AUTOMATIC DECISION"
)

print(
    "10. LONG HOLD DIAGNOSTIC"
)
